In [ ]:
!pip install torch scikit-learn scipy networkx --quiet

In [ ]:
import numpy as np
import networkx as nx
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import random

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving out.moreno_blogs_blogs to out (1).moreno_blogs_blogs


In [ ]:
G = nx.read_edgelist(
    "out.moreno_blogs_blogs",
    comments="%",
    nodetype=int,
    create_using=nx.DiGraph()
)

G = G.to_undirected()

mapping = {node: idx for idx, node in enumerate(G.nodes())}
G = nx.relabel_nodes(G, mapping)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1224
Edges: 16718


In [ ]:
N = G.number_of_nodes()

all_edges = list(G.edges())

train_edges, test_edges = train_test_split(
    all_edges,
    test_size=0.1,
    random_state=SEED
)

train_graph = nx.Graph()
train_graph.add_nodes_from(G.nodes())
train_graph.add_edges_from(train_edges)

A_train = nx.to_scipy_sparse_array(train_graph, format='csr')

print("Train edges:", len(train_edges))
print("Test edges :", len(test_edges))

Train edges: 15046
Test edges : 1672


In [ ]:
def sample_negative_edges(graph, num_samples):
    neg_edges = set()

    while len(neg_edges) < num_samples:
        u = np.random.randint(0, N)
        v = np.random.randint(0, N)

        if u != v and not graph.has_edge(u, v):
            neg_edges.add((u, v))

    return list(neg_edges)

train_neg = sample_negative_edges(train_graph, len(train_edges))
test_neg = sample_negative_edges(train_graph, len(test_edges))

In [ ]:
def build_multi_hop(A, K=3):
    mats = []

    A_power = A.copy()
    avg_deg = A.sum() / A.shape[0]

    for k in range(1, K+1):

        weighted = A_power * (1 / (avg_deg ** (k-1)))

        mats.append(
            torch.sparse_coo_tensor(
                np.vstack(weighted.nonzero()),
                weighted.data,
                weighted.shape
            ).float().to(device)
        )

        A_power = A_power @ A

    return mats

A_multi = build_multi_hop(A_train)

In [ ]:
class EdgeNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 32)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x

In [ ]:
class WSGE(nn.Module):
    def __init__(self, num_nodes):
        super().__init__()

        self.H0 = nn.Parameter(torch.randn(num_nodes, 128))

        self.W1 = nn.Linear(128, 128)
        self.W2 = nn.Linear(128, 128)
        self.W3 = nn.Linear(128, 128)

        self.edge_net = EdgeNet()
        self.classifier = nn.Linear(32, 2)

    def node_embedding(self, A):

        z1 = F.relu(self.W1(self.H0))
        z2 = F.relu(self.W2(self.H0))
        z3 = F.relu(self.W3(self.H0))

        h1 = torch.sparse.mm(A[0], z1)
        h2 = torch.sparse.mm(A[1], z2)
        h3 = torch.sparse.mm(A[2], z3)

        return h1 + h2 + h3

    def edge_embedding(self, h, edges):
        edges = torch.tensor(edges, device=device)

        hi = h[edges[:,0]]
        hj = h[edges[:,1]]

        ef = self.edge_net(torch.cat([hi, hj], dim=1))
        eb = self.edge_net(torch.cat([hj, hi], dim=1))

        return ef + eb

    def forward(self, A, edges):
        h = self.node_embedding(A)
        e = self.edge_embedding(h, edges)

        return self.classifier(e)

In [ ]:
train_data = train_edges + train_neg
test_data = test_edges + test_neg

train_labels = torch.tensor(
    [1]*len(train_edges)+[0]*len(train_neg),
    dtype=torch.long
).to(device)

test_labels = torch.tensor(
    [1]*len(test_edges)+[0]*len(test_neg),
    dtype=torch.long
).to(device)

In [ ]:
model = WSGE(N).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(100):

    model.train()
    optimizer.zero_grad()

    logits = model(A_multi, train_data)

    loss = criterion(logits, train_labels)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: {loss.item():.4f}")

Epoch 0: 9.0360
Epoch 10: 0.7211
Epoch 20: 0.5811
Epoch 30: 0.5255
Epoch 40: 0.4888
Epoch 50: 0.4722
Epoch 60: 0.4620
Epoch 70: 0.4542
Epoch 80: 0.4480
Epoch 90: 0.4427


In [ ]:
model.eval()

with torch.no_grad():
    logits = model(A_multi, test_data)

    probs = F.softmax(logits, dim=1)[:,1].cpu().numpy()
    preds = torch.argmax(logits, dim=1).cpu().numpy()

auc = roc_auc_score(test_labels.cpu(), probs)
ap = average_precision_score(test_labels.cpu(), probs)
acc = accuracy_score(test_labels.cpu(), preds)

print("AUC :", auc)
print("AP  :", ap)
print("ACC :", acc)

AUC : 0.8776953223369428
AP  : 0.8760787864118011
ACC : 0.8244617224880383
